In [ ]:
from codecarbon import EmissionsTracker
import cv2
import numpy as np
import pandas as pd
from PIL import Image
import shutil
from sklearn.metrics import matthews_corrcoef
from sklearn.model_selection import train_test_split
import time
import torch
from torch.amp import autocast, GradScaler
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms
import os
import zipfile
from sklearn.metrics import (
    matthews_corrcoef, roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve, confusion_matrix
)

import sys
sys.path.append('.')
from config import PROJECT_DIR, DATASET_DIR, DATASET_ZIP, MODELS_DIR, DEVICE, KAGGLE_DIR, PROJECT_DIR, DATASET_DIR, DATASET_ZIP, MODELS_DIR, MAPS_DIR, MAPS_ZIP, CONFIGS, LABELS, LABEL_ID, IMAGENET_MEAN, IMAGENET_STD, MODELS
from utils import build_model, ChestXrayDataset, apply_nclahe, get_stratification




This notebook downloads the VinDr-CXR dataset from Kaggle, applies the filtering pipeline and generates the final dataset with images at 1024×1024 and 256×256.

Kaggle API credentials are required to download the dataset. You can either place your `kaggle.json` file in `~/.kaggle/` or uncomment and fill in the credentials directly in the cell below.

The N_HEALTHY constant adds to the filtered dataset n 'no finding' radiographies.

The IOU_THRESHOLD constant sets the threshold of IoU consensus between radiologists' drawn BB

In [ ]:
# Set Kaggle API credentials to be able to download the dataset
# os.environ['KAGGLE_USERNAME'] = ""
# os.environ['KAGGLE_KEY'] = ""

N_HEALTHY = 4000
IOU_THRESHOLD = 0.4

In [ ]:
# Dataset xhlulu kaggle (images 1024x1024 PNG)
!kaggle datasets download -d xhlulu/vinbigdata-chest-xray-resized-png-1024x1024
!unzip -q vinbigdata-chest-xray-resized-png-1024x1024.zip -d {KAGGLE_DIR}/
!rm vinbigdata-chest-xray-resized-png-1024x1024.zip

# Official dataset VinDr-CXR (file: train.csv)
!kaggle competitions download -c vinbigdata-chest-xray-abnormalities-detection -f train.csv
!mv train.csv {KAGGLE_DIR}/train.csv 

print("Dataset VinDr-CXR 1024x1024 PNG downloaded and ready for processing.")

In [ ]:
print(f"Total of downloaded images: {len(os.listdir(f'{KAGGLE_DIR}/train'))}")

## Reescale coordinates and define needed functions


In [ ]:
# Load labels (official) y metadata (original dimensions to reescale coordenates)
df_labels = pd.read_csv(f'{KAGGLE_DIR}/train.csv')
df_meta = pd.read_csv(f'{KAGGLE_DIR}/train_meta.csv') # This one has 'width' and 'height' original dimensions

# We merge using the correct names of the dataset of xhlulu (dim1=width, dim0=height)
df_master = pd.merge(df_labels, df_meta[['image_id', 'dim0', 'dim1']], on='image_id')

# ¿are they already scaled?
if df_master['x_max'].max() > 1024:
    print("Original coordinates detected. Rescaling to 1024x1024...")

    df_master['x_min'] = df_master['x_min'] * (1024 / df_master['dim1'])
    df_master['x_max'] = df_master['x_max'] * (1024 / df_master['dim1'])
    df_master['y_min'] = df_master['y_min'] * (1024 / df_master['dim0'])
    df_master['y_max'] = df_master['y_max'] * (1024 / df_master['dim0'])

    print("Coordinates scaled successfully.")
else:
    print("Coordinates already scaled. No action needed.")

In [ ]:
def calculate_iou(box1, box2):
    """
    box: (x_min, y_min, x_max, y_max)
    """
    x_min = max(box1[0], box2[0])
    y_min = max(box1[1], box2[1])
    x_max = min(box1[2], box2[2])
    y_max = min(box1[3], box2[3])

    intersection = max(0, x_max - x_min) * max(0, y_max - y_min)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - intersection

    return intersection / union if union > 0 else 0.0

# IoU mínimo por pares entre radiólogos
def minimum_iou_between_pairs(grupo):
    """
    Given a group of rows (annotations from different radiologists on the same image and class), 
    calculate the minimum IoU between all possible pairs.
    If there is only one radiologist with consensus, return 1.0.

    """
    boxes = grupo[['x_min', 'y_min', 'x_max', 'y_max']].values
    if len(boxes) < 2:
        return 1.0
    ious = [calculate_iou(b1, b2) for b1, b2 in combinations(boxes, 2)]
    return min(ious)

## Filter dataset pipeline

In [ ]:
# Step 0: initial state dataset
print("=" * 60)
print("Step 0 - Initial state of the dataset")
print("=" * 60)
clases_tfg = [0, 3, 14]
df_interest = df_master[df_master['class_id'].isin(clases_tfg)].copy()
print(f"Total rows (annotations): {len(df_interest)}")
print(f"Unique images:             {df_interest['image_id'].nunique()}")
print()

# Step 1: Consensus filtering (≥ 2 different radiologists)
print("=" * 60)
print("Step 1 - Consensus filtering (≥ 2 different radiologists)")
print("=" * 60)

votes = (df_interest
         .groupby(['image_id', 'class_id'])['rad_id']
         .nunique()
         .reset_index()
         .rename(columns={'rad_id': 'n_votes'}))

consensus_ids = votes[votes['n_votes'] >= 2][['image_id', 'class_id']]
df_consensus = pd.merge(df_interest, consensus_ids, on=['image_id', 'class_id'])

# Only pathological classes (aneurysm and cardiomegaly) for the next steps, we will add healthy at the end
df_pathological = df_consensus[df_consensus['class_id'] != 14]

print(f"Images with semantic consensus (pathological): {df_pathological['image_id'].nunique()}")
print(df_pathological.groupby('class_name')['image_id'].nunique().to_string())
print()

# Step 2: Spatial filter (minimum IoU per pair ≥ 0.4)
print("=" * 60)
print("Step 2 — Spatial filter (minimum IoU per pair ≥ 0.4)")
print("=" * 60)

iou_por_grupo = (df_pathological
                 .groupby(['image_id', 'class_id'])
                 .apply(minimum_iou_between_pairs)
                 .reset_index()
                 .rename(columns={0: 'iou_min'}))

# Images that surpass the IoU threshold (approved) vs those that do not (discarded) 
aproved = iou_por_grupo[iou_por_grupo['iou_min'] >= IOU_THRESHOLD][['image_id', 'class_id']]
discarded_iou = iou_por_grupo[iou_por_grupo['iou_min'] < IOU_THRESHOLD][['image_id', 'class_id']]

print(f"IoU threshold: {IOU_THRESHOLD}")
print(f"Images discarded due to low spatial agreement: {len(discarded_iou)}")
print(f"  → Aortic enlargement discarded:    {len(discarded_iou[discarded_iou['class_id'] == 0])}")
print(f"  → Cardiomegaly discarded:{len(discarded_iou[discarded_iou['class_id'] == 3])}")
print()

df_aproved = pd.merge(df_pathological, aproved, on=['image_id', 'class_id'])
print(f"Images that pass the spatial filter: {df_aproved['image_id'].nunique()}")
print(df_aproved.groupby('class_name')['image_id'].nunique().to_string())
print()

# Step 3: Bounding Box Union (for each image-class, unify into a single BB that encompasses all radiologists)
print("=" * 60)
print("Step 3 — Bounding Box Union")
print("=" * 60)

final_BB = (df_aproved
            .groupby(['image_id', 'class_id', 'class_name'])
            .agg(x_min=('x_min', 'min'),
                 y_min=('y_min', 'min'),
                 x_max=('x_max', 'max'),
                 y_max=('y_max', 'max'))
            .reset_index())

print(f"Final BBs generated: {len(final_BB)}")
print(final_BB.groupby('class_name')['image_id'].nunique().to_string())
print()

# Step 4: Control group (healthy)
print("=" * 60)
print("Step 4 — Control group (healthy)")
print("=" * 60)

# Only images where all radiologists said "no finding" (class_id=14)
ids_healthy = (df_master.groupby('image_id')['class_id']
                  .apply(lambda x: (x == 14).all())
                  .reset_index()
                  .query('class_id == True')['image_id'])

df_healthy = (df_master[df_master['image_id'].isin(ids_healthy)]
            .drop_duplicates('image_id')
            .sample(n=N_HEALTHY, random_state=42))

print(f"Healthy images selected: {len(df_healthy)}")
print()

# Step 5: Final dataset
print("=" * 60)
print("Step 5 — Final dataset")
print("=" * 60)

df_1024 = pd.concat([final_BB, df_healthy[['image_id', 'class_id', 'class_name',
                                               'x_min', 'y_min', 'x_max', 'y_max']]])

# Sequential ID mapping
mapping_ids = {
    0: 0,
    3: 1,
    14: 2
}
#If you need to map the class names to another language uncomment this part and adapt the names as you wish
#mapping_names = {
#    'Aortic enlargement': 'aneurisma dell'aorta',
#    'Cardiomegaly':       'cardiomegalia',
#    'No finding':         'nessun risultato'
#}
df_1024['class_id'] = df_1024['class_id'].map(mapping_ids)
#df_1024['class_name'] = df_1024['class_name'].map(mapping_names)

# Images with both pathologies
ids_aneurysm     = set(df_1024[df_1024['class_id'] == 0]['image_id'])
ids_cardiomegaly = set(df_1024[df_1024['class_id'] == 1]['image_id'])
both = ids_aneurysm & ids_cardiomegaly

print(f"Aortic aneurysm:      {len(ids_aneurysm)}")
print(f"Cardiomegaly:  {len(ids_cardiomegaly)}")
print(f"Healthy:          {len(df_healthy)}")
print(f"Both:          {len(both)}")
print(f"Total unique images: {df_1024['image_id'].nunique()}")
print("=" * 60)

## Compress Dataset

In [ ]:
os.makedirs(f'{DATASET_DIR}/images_1024', exist_ok=True)
os.makedirs(f'{DATASET_DIR}/images_256', exist_ok=True)

print("Processing images...")

for img_id in df_1024['image_id'].unique():
    src = f'{KAGGLE_DIR}/train/{img_id}.png'
    if os.path.exists(src):
        # Save 1024x1024 version
        shutil.copy(src, f'{DATASET_DIR}/images_1024/{img_id}.png')

        # Generate and save the 256x256 version (Bicubic)
        img = Image.open(src)
        img_256 = img.resize((256, 256), resample=Image.BICUBIC)
        img_256.save(f'{DATASET_DIR}/images_256/{img_id}.png')

# Save the metadata within the same folder
df_1024.to_csv(f'{DATASET_DIR}/metadata_1024.csv', index=False)

# Create the CSV for 256x256, scaling the coordinates (factor 0.25)
df_256 = df_1024.copy()
for col in ['x_min', 'y_min', 'x_max', 'y_max']:
    df_256[col] = df_256[col] * 0.25
df_256.to_csv(f'{DATASET_DIR}/metadata_256.csv', index=False)

In [ ]:
print("Dataset processed and ready for training.")
print('01_dataset_import completed.')

## Configuration

This notebook trains AlexNet and DenseNet-121 on chest X-ray radiographs from the VinDr-CXR dataset at two resolutions (256×256 and 1024×1024) for multi-label classification of aortic enlargement and cardiomegaly.

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
df = pd.read_csv(f'{DATASET_DIR}/metadata_1024.csv')

In [ ]:
# Classify each image into a category for the split
def category_split(image_id, ids_aneurysm, ids_cardiomegaly):
    in_aneurysm = image_id in ids_aneurysm
    in_cardio = image_id in ids_cardiomegaly
    if in_aneurysm and in_cardio:
        return 'both'
    elif in_aneurysm:
        return 'aortic enlargement'
    elif in_cardio:
        return 'cardiomegaly'
    else:
        return 'healthy'

# Create DataFrame of unique images with their category 
ids_aneurysm = set(df[df['class_id'] == 0]['image_id'])
ids_cardiomegaly = set(df[df['class_id'] == 1]['image_id'])

df_unique = pd.DataFrame({'image_id': df['image_id'].unique()})
df_unique['category'] = df_unique['image_id'].apply(
    lambda x: category_split(x, ids_aneurysm, ids_cardiomegaly)
)

print("Category Distribution:")
print(df_unique['category'].value_counts())
print(f"Total unique images: {len(df_unique)}")

# Stratified split 80/10/10
train_ids, temp_ids = train_test_split(
    df_unique, test_size=0.2, stratify=df_unique['category'], random_state=42
)
val_ids, test_ids = train_test_split(
    temp_ids, test_size=0.5, stratify=temp_ids['category'], random_state=42
)

print(f"\nTrain: {len(train_ids)} images")
print(f"Val:   {len(val_ids)} images")
print(f"Test:  {len(test_ids)} images")

# Save splits as a fixed csv
train_ids[['image_id']].to_csv(f'{DATASET_DIR}/split_train.csv', index=False)
val_ids[['image_id']].to_csv(f'{DATASET_DIR}/split_val.csv', index=False)
test_ids[['image_id']].to_csv(f'{DATASET_DIR}/split_test.csv', index=False)

print("\nSplits saved")

## Reproducibility

In [ ]:
torch.manual_seed(42)
np.random.seed(42)

## Dataset Class

In [ ]:

# Dataloaders function
def create_dataloaders(metadata_csv, images_dir, resolution, batch_size):

    train_dataset = ChestXrayDataset(
        metadata_csv, f'{DATASET_DIR}/split_train.csv', images_dir, resolution, is_for_train=True
    )
    val_dataset = ChestXrayDataset(
        metadata_csv, f'{DATASET_DIR}/split_val.csv', images_dir, resolution, is_for_train=True
    )
    test_dataset = ChestXrayDataset(
        metadata_csv, f'{DATASET_DIR}/split_test.csv', images_dir, resolution, is_for_train=True
    )

    # WeightedRandomSampler to balance training set
    labels = []
    for image_id in train_dataset.image_ids:
        filas = train_dataset.df[train_dataset.df['image_id'] == image_id]
        is_clinical = (0 in filas['class_id'].values) or (1 in filas['class_id'].values)
        labels.append(1 if is_clinical else 0)

    n_clinical = sum(labels)
    n_healthy = len(labels) - n_clinical
    class_weights = [1.0 / n_healthy, 1.0 / n_clinical]
    sample_weights = [class_weights[l] for l in labels]

    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(train_dataset),
        replacement=True
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=sampler)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    print(f"Train: {len(train_dataset)} images")
    print(f"Val:   {len(val_dataset)} images")
    print(f"Test:  {len(test_dataset)} images")

    return train_loader, val_loader, test_loader

In [ ]:
def build_model(architecture, resolution, freeze_backbone=True):
    """
    architecture: 'alexnet' or 'densenet'
    resolution: 256 or 1024
    freeze_backbone: if True freezes everything except the last layer
    """
    if architecture == 'alexnet':
        model = models.alexnet(weights='IMAGENET1K_V1')
        if freeze_backbone:
            for param in model.parameters():
                param.requires_grad = False
        # Substitute final classifier
        model.classifier[6] = nn.Linear(4096, 2)

        # For higher resolution alexnet unfreeze last three feature layers
        if resolution == 1024:
            for name, param in model.named_parameters():
                if any(f'features.{i}' in name for i in [10, 11, 12]):
                    param.requires_grad = True

    elif architecture == 'densenet':
        model = models.densenet121(weights='IMAGENET1K_V1')
        if freeze_backbone:
            for param in model.parameters():
                param.requires_grad = False
        # Substitute final classifier
        model.classifier = nn.Linear(1024, 2)

        # For higher resolution densenet-121 unfreeze last dense block and norm layer
        if resolution == 1024:
            for name, param in model.named_parameters():
                if 'denseblock4' in name or 'norm5' in name:
                    param.requires_grad = True

    return model.to(DEVICE)


def train_model(model, train_loader, val_loader, architecture, resolution,
                    epochs=30, patience=5):

    lr = 1e-4 if resolution == 256 else 1e-5
    optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    criterion = nn.BCEWithLogitsLoss()
    scaler = GradScaler()  # Mixed precision

    best_val_loss = float('inf')
    epochs_without_improvement = 0
    record = {'train_loss': [], 'val_loss': []}

    for epoch in range(epochs):
        # Train
        model.train()
        train_loss = 0
        t0 = time.time()

        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE
            )
            optimizer.zero_grad()

            with autocast(DEVICE.type):
                outputs = model(imgs)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item()

        train_loss /= len(train_loader)

        # Validation
        model.eval()
        val_loss = 0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                with autocast():
                    outputs = model(imgs)
                    loss = criterion(outputs, labels)
                val_loss += loss.item()
                all_preds.append((outputs.cpu().numpy() > 0.5).astype(int))
                all_labels.append(labels.cpu().numpy())

        val_loss /= len(val_loader)
        all_preds = np.concatenate(all_preds)
        all_labels = np.concatenate(all_labels)

        mcc_aneurysm = matthews_corrcoef(all_labels[:, 0], all_preds[:, 0])
        mcc_cardio = matthews_corrcoef(all_labels[:, 1], all_preds[:, 1])

        print(f"Epoch {epoch+1}/{epochs} | "
        f"Train: {train_loss:.4f} | Val: {val_loss:.4f} | "
        f"MCC [Aneu: {mcc_aneurysm:.3f} Card: {mcc_cardio:.3f}] | "
        f"Time: {time.time()-t0:.1f}s")

        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(),
                      f'{MODELS_DIR}/{architecture}_{resolution}_best.pth')
            epochs_without_improvement = 0
            print(f"   ✓ Best model saved")
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f"   Early stopping in epoch {epoch+1}")
                break

    return record

## AlexNet 256x256


In [ ]:
# Create dataloaders
train_loader, val_loader, test_loader = create_dataloaders(
    metadata_csv=f'{DATASET_DIR}/metadata_256.csv',
    images_dir=f'{DATASET_DIR}/images_256',
    resolution=256,
    batch_size=16
)

In [ ]:
# Build AlexNet
model = build_model('alexnet', resolution=256)

# Verify trainable parameters and freezed parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable_params:,} / {total_params:,}")

# Train and monitor AlexNet with 256x256 images
tracker = EmissionsTracker()
tracker.start()

record = train_model(model, train_loader, val_loader,
                        architecture='alexnet', resolution=256)

emissions = tracker.stop()
print(f"CO2 emissions: {emissions:.6f} kg")

## DenseNet-121 256x256

In [ ]:
train_loader, val_loader, test_loader = create_dataloaders(
    metadata_csv=f'{DATASET_DIR}/metadata_256.csv',
    images_dir=f'{DATASET_DIR}/images_256',
    resolution=256,
    batch_size=16
)

In [ ]:

model = build_model('densenet', resolution=256)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable_params:,} / {total_params:,}")

tracker = EmissionsTracker()
tracker.start()

record = train_model(model, train_loader, val_loader,
                        architecture='densenet', resolution=256, epochs=50)

emissions = tracker.stop()
print(f"CO2 emissions: {emissions:.6f} kg")

## AlexNet 1024x1024


In [ ]:
train_loader, val_loader, test_loader = create_dataloaders(
    metadata_csv=f'{DATASET_DIR}/metadata_1024.csv',
    images_dir=f'{DATASET_DIR}/images_1024',
    resolution=1024,
    batch_size=4
)

In [ ]:
model = build_model('alexnet', resolution=1024)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable_params:,} / {total_params:,}")

tracker = EmissionsTracker()
tracker.start()

record = train_model(model, train_loader, val_loader,
                        architecture='alexnet', resolution=1024,
                            epochs=50, patience=5)

emissions = tracker.stop()
print(f"CO2 emissions: {emissions:.6f} kg")

## DenseNet-121 1024x1024

In [ ]:
train_loader, val_loader, test_loader = create_dataloaders(
    metadata_csv=f'{DATASET_DIR}/metadata_1024.csv',
    images_dir=f'{DATASET_DIR}/images_1024',
    resolution=1024,
    batch_size=4
)

In [ ]:
model = build_model('densenet', resolution=1024)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable_params:,} / {total_params:,}")

tracker = EmissionsTracker(log_level="error")
tracker.start()

record = train_model(model, train_loader, val_loader,
                        architecture='densenet', resolution=1024,
                            epochs=50, patience=5)

emissions = tracker.stop()
print(f"CO2 emissions: {emissions:.6f} kg")

In [ ]:
print('Dataset folder deleted.')
print('02_preprocessing_images_and_training completed.')

## 03_models_clinical_validation

In [ ]:
# Clinical evaluation function for each model

def clinical_eval_model(model, test_loader):
    model.eval()
    all_logits = []
    all_labels = []

    with torch.no_grad():
        for imgs, labels, _ in test_loader:
            imgs = imgs.to(DEVICE)
            outputs = model(imgs)
            # Save logits (before sigmoid) for AUC
            all_logits.append(outputs.cpu().numpy())
            all_labels.append(labels.numpy())

    all_logits = np.concatenate(all_logits)     # (N, 2)
    all_labels = np.concatenate(all_labels)     # (N, 2)
    all_probs  = 1 / (1 + np.exp(-all_logits))  # sigmoid

    results = {}

    for i, name in enumerate(LABELS):
        y_true = all_labels[:, i]
        y_prob = all_probs[:, i]
    
        # Optimal threshold by MCC over ROC 
        fpr, tpr, thresholds = roc_curve(y_true, y_prob)
        mccs = []
        for t in thresholds:
            y_pred_t = (y_prob >= t).astype(int)
            mccs.append(matthews_corrcoef(y_true, y_pred_t))
        optimal_threshold = thresholds[np.argmax(mccs)]
        y_pred = (y_prob >= optimal_threshold).astype(int)

        # Clinical metrics
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        mcc = matthews_corrcoef(y_true, y_pred)
        auc_roc = roc_auc_score(y_true, y_prob)
        pr_auc  = average_precision_score(y_true, y_prob)

        results[name] = {
            'optimal threshold': round(optimal_threshold, 4),
            'MCC':               round(mcc, 4),
            'AUC-ROC':           round(auc_roc, 4),
            'PR-AUC':            round(pr_auc, 4),
            'recall':            round(recall, 4),
            'specificity':       round(specificity, 4),
            'TP': int(tp), 'TN': int(tn), 'FP': int(fp), 'FN': int(fn)
        }

    return results

In [ ]:
all_results = {}
global_thresholds = {}

for cfg in CONFIGS:

    print(f"\n{'='*50}")
    print(f"Evaluating {cfg['name']}...")

    # Dataset and loader
    test_dataset = ChestXrayDataset(
        metadata_csv=f"{DATASET_DIR}/{cfg['metadata']}",
        split_csv=f"{DATASET_DIR}/split_test.csv",
        images_dir=f"{DATASET_DIR}/{cfg['images']}",
        resolution=cfg['res']
    )
    test_loader = DataLoader(test_dataset, batch_size=cfg['batch'], shuffle=False)
    print(f"Test set: {len(test_dataset)} images")

    # Load model
    model = build_model(cfg['arq'], DEVICE)
    pth_path = f"{MODELS_DIR}/{cfg['arq']}_{cfg['res']}_best.pth"
    model.load_state_dict(torch.load(pth_path, map_location=DEVICE))
    print(f"Checkpoint loaded: {pth_path}")

    # Evaluate
    results = clinical_eval_model(model, test_loader)
    global_thresholds[cfg['name']] = {
    'aneurysm': results['aneurysm']['optimal threshold'],
    'cardiomegaly': results['cardiomegaly']['optimal threshold']
    }
    all_results[cfg['name']] = results

    # Print results
    for label, metrics in results.items():
        print(f"\n  {label}:")
        for k, v in metrics.items():
            print(f"    {k}: {v}")

print("\n✓ Evaluation completed.")

In [ ]:
# Comparative summary table
rows = []
for model, labels in all_results.items():
    for label, metrics in labels.items():
        rows.append({
            'Model': model,
            'Label': label,
            'MCC': metrics['MCC'],
            'AUC-ROC': metrics['AUC-ROC'],
            'PR-AUC': metrics['PR-AUC'],
            'Recall': metrics['recall'],
            'Specificity': metrics['specificity'],
            'Threshold': metrics['optimal threshold']
        })

df_results = pd.DataFrame(rows)
print(df_results.to_string(index=False))

# Save to PROJECT_DIR
df_results.to_csv(f'{PROJECT_DIR}/results_clinical_evaluation.csv', index=False)
print(f'\n✓ Saved: {PROJECT_DIR}/results_clinical_evaluation.csv')

In [ ]:
# Save predictions by image
registers = []

for cfg in CONFIGS:
    test_dataset = ChestXrayDataset(
        metadata_csv=f"{DATASET_DIR}/{cfg['metadata']}",
        split_csv=f"{DATASET_DIR}/split_test.csv",
        images_dir=f"{DATASET_DIR}/{cfg['images']}",
        resolution=cfg['res']
    )
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

    model = build_model(cfg['arq'], DEVICE)
    pth_path = f"{MODELS_DIR}/{cfg['arq']}_{cfg['res']}_best.pth"
    model.load_state_dict(torch.load(pth_path, map_location=DEVICE))
    model.eval()

    u = global_thresholds[cfg['name']]

    with torch.no_grad():
        for imgs, labels, image_ids in test_loader:
            imgs = imgs.to(DEVICE)
            output = model(imgs)
            probs = torch.sigmoid(output).cpu().numpy()[0]
            label = labels.numpy()[0]
            image_id = image_ids[0]

            registers.append({
                'model': cfg['name'],
                'image_id': image_id,
                'prob_aneurysm': round(float(probs[0]), 4),
                'prob_cardiomegaly': round(float(probs[1]), 4),
                'pred_aneurysm': int(probs[0] >= u['aneurysm']),
                'pred_cardiomegaly': int(probs[1] >= u['cardiomegaly']),
                'label_aneurysm': int(label[0]),
                'label_cardiomegaly': int(label[1]),
            })

df_preds = pd.DataFrame(registers)
df_preds.to_csv(f"{PROJECT_DIR}/predictions_by_image.csv", index=False)
print(f'✓ Saved: {len(df_preds)} rows to predictions_by_image.csv')

## Approach to clinical valid threshold

In [ ]:
print("Minimum lower bound to have FN=0 in aortic enlargement by model:\n")

for model in ['AN_256', 'DN_256', 'AN_1024', 'DN_1024']:
    df_m = df_preds[df_preds['model'] == model].copy()

    # Only images with real aortic enlargement
    df_aneu = df_m[df_m['label_aneurysm'] == 1]

    # Minimum threshold = minimum probability among true positives
    threshold_fn0 = df_aneu['prob_aneurysm'].min()

    # With that threshold, calculate FP and specificity
    df_m['pred_new'] = (df_m['prob_aneurysm'] >= threshold_fn0).astype(int)

    TN = ((df_m['pred_new'] == 0) & (df_m['label_aneurysm'] == 0)).sum()
    FP = ((df_m['pred_new'] == 1) & (df_m['label_aneurysm'] == 0)).sum()
    TP = ((df_m['pred_new'] == 1) & (df_m['label_aneurysm'] == 1)).sum()
    FN = ((df_m['pred_new'] == 0) & (df_m['label_aneurysm'] == 1)).sum()

    specificity = TN / (TN + FP) if (TN + FP) > 0 else 0
    recall  = TP / (TP + FN) if (TP + FN) > 0 else 0

    print(f'{model}:')
    print(f'  Minimum threshold FN=0: {threshold_fn0:.4f}')
    print(f'  Recall: {recall:.4f} | Specificity: {specificity:.4f}')
    print(f'  TP: {TP} | FP: {FP} | TN: {TN} | FN: {FN}\n')

In [ ]:
print('Clinical validation of the models done')

## 04_saliency_maps

In [ ]:
os.makedirs(f'{MAPS_DIR}', exist_ok=True)
print('.')

In [ ]:
def get_target_layer(model, architecture):
    if architecture == 'alexnet':
        return model.features[12]
    elif architecture == 'densenet':
        return model.features.denseblock4

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.gradients = None
        self.activations = None
        target_layer.register_forward_hook(self._save_activations)
        target_layer.register_full_backward_hook(self._save_gradients)

    def _save_activations(self, module, input, output):
        self.activations = output.detach().clone()

    def _save_gradients(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach().clone()

    def generate(self, input_tensor, class_idx):
        input_tensor = input_tensor.requires_grad_(True)
        self.model.zero_grad()
        output = self.model(input_tensor)
        output[0, class_idx].backward(retain_graph=True)
        weights = self.gradients.mean(dim=[2, 3], keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = torch.relu(cam.clone())
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam

In [ ]:
# Verify resolution of saliency maps
for arq, res in [('alexnet', 256), ('alexnet', 1024), ('densenet', 256), ('densenet', 1024)]:
    if arq == 'alexnet':
        model = models.alexnet(weights=None)
        model.classifier[6] = nn.Linear(4096, 2)
        target = model.features[12]
    else:
        model = models.densenet121(weights=None)
        model.classifier = nn.Linear(1024, 2)
        target = model.features.norm5

    activation = {}
    def hook(module, input, output):
        activation['out'] = output

    target.register_forward_hook(hook)
    x = torch.zeros(1, 3, res, res)
    model(x)
    print(f"{arq} {res}x{res}: {activation['out'].shape}")

In [ ]:
for cfg in CONFIGS:
    name = cfg['name']
    print(f"\n{'='*50}")
    print(f"Generating maps for {name}...")

    out_dir = f'{MAPS_DIR}/{name}'
    os.makedirs(out_dir, exist_ok=True)

    dataset = ChestXrayDataset(
        metadata_csv=cfg['metadata'],
        split_csv=f'{DATASET_DIR}/split_test.csv',
        images_dir=cfg['images'],
        resolution=cfg['res']
    )
    loader = DataLoader(dataset, batch_size=1, shuffle=False)

    model = build_model(cfg['arq'], DEVICE)
    pth = f"{MODELS_DIR}/{cfg['arq']}_{cfg['res']}_best.pth"
    model.load_state_dict(torch.load(pth, map_location=DEVICE))
    model.eval()


    for module in model.modules():
        module._forward_hooks.clear()
        module._backward_hooks.clear()
        module._forward_pre_hooks.clear()

    target_layer = get_target_layer(model, cfg['arq'])
    gcam = GradCAM(model, target_layer=get_target_layer(model, cfg['arq']))

    for i, (img, label, image_id) in enumerate(loader):
        img = img.to(DEVICE)
        image_id = image_id[0]

        maps = {}
        for class_idx, label in enumerate(LABELS):
            cam =  gcam.generate(img, class_idx)
            res = cfg['res']
            maps[f'gradcam_{label}']   = cv2.resize(cam,   (res, res)).astype(np.float32)

        np.savez_compressed(f'{out_dir}/{image_id}.npz', **maps)

        if (i + 1) % 100 == 0:
            print(f'  {i+1}/{len(dataset)} processed')

    print(f'  {name} completed.')

print('\nAll models completed.')

## Restore

In [ ]:
print('Compressing...')
with zipfile.ZipFile(f'{MAPS_ZIP}', 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(f'{MAPS_DIR}'):
        for file in files:
            filepath = os.path.join(root, file)
            arcname = os.path.relpath(filepath, f'{MAPS_DIR}')
            zf.write(filepath, arcname)

print('Done.')
print('03_saliency_maps completed.')

## 05_maps_clinical_validation

In [ ]:
THRESHOLD_ACTIVATION = 0.2 # Threshold for binarizing activation maps when computing IoU.

In [ ]:
# Loading predictions by image
df_preds = pd.read_csv(f'{{PROJECT_DIR}}/predictions_by_image.csv')
print(f'Predictions loaded: {len(df_preds)} rows')
print(df_preds.head())

In [ ]:
# Saliency metrics
def pointing_game(cam, x_min, y_min, x_max, y_max):
    """1 if the maximum activation pixel falls within the BB, 0 otherwise."""
    max_idx = np.unravel_index(cam.argmax(), cam.shape)
    py, px = max_idx
    return int(x_min <= px <= x_max and y_min <= py <= y_max)


def poe(cam, x_min, y_min, x_max, y_max):
    """Proportion of energy of the map within the BB."""
    total = cam.sum()
    if total == 0:
        return 0.0
    x_min, y_min, x_max, y_max = int(x_min), int(y_min), int(x_max), int(y_max)
    dentro = cam[y_min:y_max, x_min:x_max].sum()
    return float(dentro / total)


# Heatmap/pathological anatomy convergence metrics - BB (requires binarization of the saliency map)

def binarize_map(map, threshold=0.20):
    """Binarizes the heatmap with a threshold over the maximum value."""
    return (map >= threshold * map.max()).astype(np.float32)


def dsc(cam_bin, x_min, y_min, x_max, y_max, res):
    """Dice Similarity Coefficient between binarized map and BB mask."""
    x_min, y_min, x_max, y_max = int(x_min), int(y_min), int(x_max), int(y_max)
    mask = np.zeros((res, res), dtype=np.float32)
    mask[y_min:y_max, x_min:x_max] = 1.0
    intersection = (cam_bin * mask).sum()
    suma = cam_bin.sum() + mask.sum()
    if suma == 0:
        return 0.0
    return float(2 * intersection / suma)


def iou_between_maps(cam1, cam2, threshold=0.20):
    """IoU between two binarized heatmaps (discrimination between labels)."""
    bin1 = binarize_map(cam1, threshold)
    bin2 = binarize_map(cam2, threshold)
    intersection = (bin1 * bin2).sum()
    union = ((bin1 + bin2) > 0).sum()
    if union == 0:
        return 0.0
    return float(intersection / union)

In [ ]:
# Deletion AUC and Insertion AUC

def deletion_insertion_auc(model, img_tensor, cam, class_idx, n_steps=10):
    """
    Calculates Deletion AUC and Insertion AUC.
     - Deletion: progressively removes the most important pixels
     - Insertion: progressively introduces the most important pixels
    Returns:
      (deletion_auc, insertion_auc)
    """
    model.eval()
    img_np = img_tensor.squeeze().cpu().numpy()  # (3, H, W)
    H, W = img_np.shape[1], img_np.shape[2]

    # Sort pixels by importance (highest to lowest)
    cam_flat = cam.flatten()
    sort = np.argsort(cam_flat)[::-1]

    # Reference image for insetion (blurred image)
    blur_img = cv2.GaussianBlur(
        img_np.transpose(1, 2, 0),
        (51, 51), 0
    ).transpose(2, 0, 1)

    deletion_scores = []
    insertion_scores = []

    n_pixels = H * W
    steps = [int(n_pixels * i / n_steps) for i in range(n_steps + 1)]

    img_del = img_np.copy().reshape(3, -1)
    img_ins = blur_img.copy().reshape(3, -1)

    for step in steps:
        # Deletion: put to zero the most important pixels
        img_del_step = img_np.copy().reshape(3, -1)
        img_del_step[:, sort[:step]] = 0
        tensor_del = torch.FloatTensor(img_del_step.reshape(3, H, W)).unsqueeze(0).to(DEVICE)

        # Insertion: reveal the most important pixels from the blurred image
        img_ins_step = blur_img.copy().reshape(3, -1)
        img_ins_step[:, sort[:step]] = img_np.reshape(3, -1)[:, sort[:step]]
        tensor_ins = torch.FloatTensor(img_ins_step.reshape(3, H, W)).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            prob_del = torch.sigmoid(model(tensor_del))[0, class_idx].item()
            prob_ins = torch.sigmoid(model(tensor_ins))[0, class_idx].item()

        deletion_scores.append(prob_del)
        insertion_scores.append(prob_ins)

    x = np.linspace(0, 1, n_steps + 1)
    del_auc = auc(x, deletion_scores)
    ins_auc = auc(x, insertion_scores)

    return del_auc, ins_auc

In [ ]:
# Main

registers = []

for cfg in CONFIGS:
    name = cfg['name']
    res = cfg['res']
    print(f"\n{'='*50}")
    print(f"Calculating metrics for {name}...")

    # Metadata with BBs
    df_meta = pd.read_csv(cfg['metadata'])
    df_meta = df_meta[df_meta['class_id'].isin([0, 1])]  # only pathological classes
    # Predictions of this model
    df_pred_model = df_preds[df_preds['model'] == name]

    # Load model for Del/Ins AUC
    model = build_model(cfg['arq'], DEVICE)
    pth = f"{MODELS_DIR}/{cfg['arq']}_{res}_best.pth"
    model.load_state_dict(torch.load(pth, map_location=DEVICE))
    model.eval()
    for module in model.modules():
        if isinstance(module, torch.nn.ReLU):
            module.inplace = False

    # Normalization for reconstructing tensor
    normalize = transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )

    maps_dir = f'{MAPS_DIR}/{name}'
    image_ids = df_pred_model['image_id'].unique()

    for i, image_id in enumerate(image_ids):
        # Load npz maps
        npz_path = f'{maps_dir}/{image_id}.npz'
        if not os.path.exists(npz_path):
            continue
        maps = np.load(npz_path)

        # Load original image and convert to tensor
        if res == 256:
            img_path = f'{DATASET_DIR}/images_256/{image_id}.png'
        else:
            img_path = f'{DATASET_DIR}/images_1024/{image_id}.png'

        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            print(f'Not found: {img_path}')
            continue
       
        tile_size = 4 if res == 256 else 16
        img_clahe = apply_nclahe(img)
        img_rgb = cv2.cvtColor(img_clahe, cv2.COLOR_GRAY2RGB)
        img_tensor = normalize(transforms.ToTensor()(Image.fromarray(img_rgb))).unsqueeze(0)

        # Prediction of this model for this image
        pred_row = df_pred_model[df_pred_model['image_id'] == image_id]
        if len(pred_row) == 0:
            continue
        pred_row = pred_row.iloc[0]

        for label_name in LABELS:
            class_idx = LABEL_ID[label_name]
            cam_key = f'gradcam_{label_name}'
            if cam_key not in maps:
                continue
            cam = maps[cam_key]  # (res, res), float32, normalized [0,1]

            # Real label and prediction
            label_col = f'label_{label_name}'
            pred_col = f'pred_{label_name}'
            label = int(pred_row[label_col])
            pred = int(pred_row[pred_col])

            # Classify TP/FP/FN/TN
            if label == 1 and pred == 1:
                stratification = 'TP'
            elif label == 0 and pred == 1:
                stratification = 'FP'
            elif label == 1 and pred == 0:
                stratification = 'FN'
            else:
                stratification = 'TN'

            # BB of this image and label (if exists)
            bb_rows = df_meta[
                (df_meta['image_id'] == image_id) &
                (df_meta['class_id'] == class_idx)
            ]
            has_bb = len(bb_rows) > 0

            pg = np.nan
            poe_val = np.nan
            dsc_val = np.nan

            if has_bb:
                bb = bb_rows.iloc[0]
                x_min, y_min = bb['x_min'], bb['y_min']
                x_max, y_max = bb['x_max'], bb['y_max']

                pg = pointing_game(cam, x_min, y_min, x_max, y_max)
                poe_val = poe(cam, x_min, y_min, x_max, y_max)
                cam_bin = binarize_map(cam)
                dsc_val = dsc(cam_bin, x_min, y_min, x_max, y_max, res)
            # IoU between maps of both labels (only if both exist)
            iou_maps = np.nan
            cam_other_key = 'gradcam_cardiomegaly' if label_name == 'aneurysm' else 'gradcam_aneurysm'
            if cam_other_key in maps.files:
                cam_other = maps[cam_other_key]
                iou_maps = iou_between_maps(cam, cam_other)

            # Del and Ins AUC
            del_auc, ins_auc = deletion_insertion_auc(
                model, img_tensor, cam, class_idx, n_steps=10
            )

            registers.append({
                'model': name,
                'image_id': image_id,
                'label': label_name,
                'stratification': stratification,  # TP/FP/FN/TN
                'pred': pred,
                'pointing_game': pg,
                'poe': poe_val,
                'dsc': dsc_val,
                'iou_between_maps': iou_maps,
                'del_auc': del_auc,
                'ins_auc': ins_auc,
            })

        if (i + 1) % 100 == 0:
            print(f'  {i+1}/{len(image_ids)} processed')

    print(f'  {name} completed.')

df_metrics = pd.DataFrame(registers)
df_metrics.to_csv(f'{PROJECT_DIR}/xai_metrics.csv', index=False)
print(f'\n Saved: {len(df_metrics)} rows in xai_metrics.csv')

In [ ]:
df = pd.read_csv(f'{PROJECT_DIR}/xai_metrics.csv')

print('\n=== Global metrics ===')
global_agg = df.groupby(['model', 'label']).agg(
    pointing_game=('pointing_game', 'mean'),
    poe=('poe', 'mean'),
    dsc=('dsc', 'mean'),
    iou_between_maps=('iou_between_maps', 'mean'),
    del_auc=('del_auc', 'mean'),
    ins_auc=('ins_auc', 'mean'),
    n=('image_id', 'count')
).round(4)
print(global_agg.to_string())

print('\n=== Metrics by stratification (TP/FP/FN) ===')
tipo_agg = df[df['stratification'] != 'TN'].groupby(['model', 'label', 'stratification']).agg(
    pointing_game=('pointing_game', 'mean'),
    poe=('poe', 'mean'),
    dsc=('dsc', 'mean'),
    del_auc=('del_auc', 'mean'),
    ins_auc=('ins_auc', 'mean'),
    n=('image_id', 'count')
).round(4)
print(tipo_agg.to_string())

# Save summary tables
global_agg.to_csv(f'{PROJECT_DIR}/global_xai_summary.csv')
tipo_agg.to_csv(f'{PROJECT_DIR}/stratification_xai_summary.csv')
print('\nTables saved')

## IoU between maps of both labels

In [ ]:
df_preds = pd.read_csv('{PROJECT_DIR}/predicciones_por_imagen.csv')

df_preds['type_aneurysm']     = df_preds.apply(lambda r: get_stratification(r, 'aneurisma'), axis=1)
df_preds['type_cardiomegalia'] = df_preds.apply(lambda r: get_stratification(r, 'cardiomegalia'), axis=1)
df_preds['combo']              = df_preds['type_aneurysm'] + '/' + df_preds['type_cardiomegalia']

results = []

for model in MODELS:
    print(f'Processing {model}...')
    df_m = df_preds[df_preds['modelo'] == model]

    for _, row in df_m.iterrows():
        image_id = row['image_id']
        combo    = row['combo']

        path = os.path.join(MAPS_DIR, model, f'{image_id}.npz')
        if not os.path.exists(path):
            continue

        data    = np.load(path)
        mapa_an = data['gradcam_aneurisma']
        mapa_ca = data['gradcam_cardiomegalia']

        iou = iou_between_maps(mapa_an, mapa_ca)

        results.append({
            'model':   model,
            'image_id': image_id,
            'combo':    combo,
            'iou':      iou
        })

df_iou = pd.DataFrame(results)

table = df_iou.groupby(['model', 'combo'])['iou'].agg(['mean', 'count']).round(4)
table.columns = ['IoU medio', 'n']
print('\nAverage IoU between maps by model and verdict combo:\n')
print(table.to_string())

df_iou.to_csv(f'{PROJECT_DIR}/iou_estratificado_combo.csv', index=False)
table.to_csv(f'{PROJECT_DIR}/iou_estratificado_resumen.csv', index=False)
print('\Saved.')

## it1_black_pixels_ratio

In [ ]:
IMG_256_DIR = f'{DATASET_DIR}/images_256'
IMG_1024_DIR = f'{DATASET_DIR}/images_1024'
THRESHOLD = 10

## Functions 

In [ ]:

def count_zero_padding_borders(img, threshold=THRESHOLD):
    """
    Counts black rows/columns from each border.
    Args:
        img: Grayscale image as a 2D numpy array.
        threshold: Pixel intensity threshold to consider a row/column as black. Permitted values grayscale: 0-255.
    Returns:
        top, bot, left, right: Number of black rows/columns from each border.
    """

    top = 0
    for i in range(img.shape[0]):
        if img[i, :].mean() < threshold:
            top += 1
        else:
            break

    bot = 0
    for i in range(img.shape[0]-1, -1, -1):
        if img[i, :].mean() < threshold:
            bot += 1
        else:
            break

    left = 0
    for j in range(img.shape[1]):
        if img[:, j].mean() < threshold:
            left += 1
        else:
            break

    right = 0
    for j in range(img.shape[1]-1, -1, -1):
        if img[:, j].mean() < threshold:
            right += 1
        else:
            break

    return top, bot, left, right

In [ ]:
def dark_pixels_ratio(img, threshold=THRESHOLD):
    return (img < threshold).sum() / img.size

In [ ]:
# Calculate dark pixels ratio for all images once
dark_ratios_256 = {}
for fname in os.listdir(IMG_256_DIR):
    if not fname.endswith('.png'):
        continue
    image_id = fname.replace('.png', '')
    img = cv2.imread(os.path.join(IMG_256_DIR, fname), cv2.IMREAD_GRAYSCALE)
    if img is None:
        continue
    dark_ratios_256[image_id] = dark_pixels_ratio(img)

dark_ratios_1024 = {}
for fname in os.listdir(IMG_1024_DIR):
    if not fname.endswith('.png'):
        continue
    image_id = fname.replace('.png', '')
    img = cv2.imread(os.path.join(IMG_1024_DIR, fname), cv2.IMREAD_GRAYSCALE)
    if img is None:
        continue
    dark_ratios_1024[image_id] = dark_pixels_ratio(img)

df_dark_256  = pd.DataFrame(list(dark_ratios_256.items()),  columns=['image_id', 'dark_ratio'])
df_dark_1024 = pd.DataFrame(list(dark_ratios_1024.items()), columns=['image_id', 'dark_ratio'])

In [ ]:
# Dark pixels ratio by class for each resolution
dark_pixels_records = []

for res, df_dark, metadata_csv in [
    ('256',  df_dark_256,  f'{DATASET_DIR}/metadata_256.csv'),
    ('1024', df_dark_1024, f'{DATASET_DIR}/metadata_1024.csv')
]:
    print(f'\n=== {res}x{res} ===')
    df_meta = pd.read_csv(metadata_csv)
    df_all = df_meta.drop_duplicates('image_id').merge(df_dark, on='image_id', how='left')

    ids_aneurysm      = set(df_all[df_all['class_id'] == 0]['image_id'])
    ids_cardiomegaly  = set(df_all[df_all['class_id'] == 1]['image_id'])
    ids_healthy       = set(df_all[df_all['class_id'] == 2]['image_id'])
    ids_both          = ids_aneurysm & ids_cardiomegaly
    ids_only_aneurysm = ids_aneurysm - ids_cardiomegaly
    ids_only_cardio   = ids_cardiomegaly - ids_aneurysm

    classes = {
        'Healthy':           ids_healthy,
        'Only aneurysm':     ids_only_aneurysm,
        'Only cardiomegaly': ids_only_cardio,
        'Both':              ids_both
    }

    for name, ids in classes.items():
        df_r = df_dark[df_dark['image_id'].isin(ids)]
        mean = df_r['dark_ratio'].mean()
        std  = df_r['dark_ratio'].std()
        print(f'{name} (n={len(ids)}): mean={mean:.4f} | std={std:.4f}')
        dark_pixels_records.append({
            'resolution': res,
            'class': name,
            'n': len(ids),
            'mean_dark_ratio': round(mean, 4),
            'std_dark_ratio': round(std, 4)
        })

df_dark_pixels = pd.DataFrame(dark_pixels_records)
df_dark_pixels.to_csv(f'{PROJECT_DIR}/dark_pixels_analysis.csv', index=False)
print('\nSaved: dark_pixels_analysis.csv')

In [ ]:
# Zero padding ratio by class for each resolution
results = []

for res, img_dir, metadata_csv in [
    ('256',  IMG_256_DIR,  f'{DATASET_DIR}/metadata_256.csv'),
    ('1024', IMG_1024_DIR, f'{DATASET_DIR}/metadata_1024.csv')
]:
    print(f'\nProcessing {res}x{res}...')
    df_meta = pd.read_csv(metadata_csv)
    files = os.listdir(img_dir)

    ratios = {}
    for f in files:
        img = cv2.imread(os.path.join(img_dir, f), cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
        image_id = f.replace('.png', '')
        top, bot, left, right = count_zero_padding_borders(img)
        total_px = img.shape[0] * img.shape[1]
        padding_px = (top + bot) * img.shape[1] + (left + right) * img.shape[0]
        ratios[image_id] = padding_px / total_px
        results.append({
            'image_id': image_id,
            'resolution': res,
            'top': top,
            'bottom': bot,
            'left': left,
            'right': right,
            'padding_ratio': padding_px / total_px
        })
    print(f'  {len(files)} images processed.')

    df_padding = pd.DataFrame(list(ratios.items()), columns=['image_id', 'padding_ratio'])
    df_all = df_meta.drop_duplicates('image_id').merge(df_padding, on='image_id', how='left')

    ids_aneurysm      = set(df_all[df_all['class_id'] == 0]['image_id'])
    ids_cardiomegaly  = set(df_all[df_all['class_id'] == 1]['image_id'])
    ids_healthy       = set(df_all[df_all['class_id'] == 2]['image_id'])
    ids_both          = ids_aneurysm & ids_cardiomegaly
    ids_only_aneurysm = ids_aneurysm - ids_cardiomegaly
    ids_only_cardio   = ids_cardiomegaly - ids_aneurysm

    classes = {
        'Healthy':           ids_healthy,
        'Only aneurysm':     ids_only_aneurysm,
        'Only cardiomegaly': ids_only_cardio,
        'Both':              ids_both
    }

    print(f'\n=== {res}x{res} — Zero padding by class ===')
    for name, ids in classes.items():
        df_r = df_padding[df_padding['image_id'].isin(ids)]
        print(f'{name} (n={len(ids)}): mean={df_r["padding_ratio"].mean():.4f} | std={df_r["padding_ratio"].std():.4f}')

df_ratios = pd.DataFrame(results)
for res in RESOLUTIONS:
    df_r = df_ratios[df_ratios['resolution'] == res]
    print(f'\n=== {res}x{res} — Border stats ===')
    for col in ['top', 'bottom', 'left', 'right', 'padding_ratio']:
        print(f'{col}: mean={df_r[col].mean():.3f}, max={df_r[col].max():.1f}')

df_ratios.to_csv(f'{PROJECT_DIR}/zero_padding_analysis.csv', index=False)
print('\nSaved.')

In [ ]:
# Dark pixels ratio by model, label and prediction type
df_preds = pd.read_csv(f'{PROJECT_DIR}/predictions_by_image.csv')
df_preds['type_aneurysm']     = df_preds.apply(lambda r: get_stratification(r, 'aneurysm'), axis=1)
df_preds['type_cardiomegaly'] = df_preds.apply(lambda r: get_stratification(r, 'cardiomegaly'), axis=1)

print('\nMean dark pixels ratio by prediction type:\n')
for label in LABELS:
    col_type = f'type_{label}'
    print(f'── {label.upper()} ──')
    for model in MODELS:
        res = model.split('_')[-1]
        df_dark = df_dark_256 if res == '256' else df_dark_1024
        df_m = df_preds[df_preds['model'] == model].merge(df_dark, on='image_id', how='left')
        summary = df_m.groupby(col_type)['dark_ratio'].mean().round(4)
        print(f'  {model}: {summary.to_dict()}')
    print()

In [ ]:
# Optimal dark pixels threshold as verdict classifier
print("Optimal dark pixels threshold as verdict classifier:\n")
for label in LABELS:
    print(f'── {label.upper()} ──')
    for model in MODELS:
        res = model.split('_')[-1]
        df_dark = df_dark_256 if res == '256' else df_dark_1024
        df_m = df_preds[df_preds['model'] == model].merge(df_dark, on='image_id', how='left').dropna(subset=['dark_ratio'])

        X = df_m['dark_ratio'].values
        y = df_m[f'pred_{label}'].values

        thresholds = np.linspace(X.min(), X.max(), 1000)
        best_acc = 0
        best_threshold = 0
        for t in thresholds:
            pred = (X >= t).astype(int)
            acc = (pred == y).mean()
            if acc > best_acc:
                best_acc = acc
                best_threshold = t

        auc = roc_auc_score(y, X)
        print(f'  {model}: threshold={best_threshold:.4f} | acc={best_acc:.3f} | AUC={auc:.3f}')
    print()

In [ ]:
print('black pixels ratio analyzed.')

## Restore

In [ ]:
print(f'Removing {DATASET_DIR} and {MAPS_DIR}...')
shutil.rmtree(DATASET_DIR)
shutil.rmtree(MAPS_DIR)